# Membina Aplikasi Penjanaan Imej (OpenAI)

LLM lebih daripada hanya penjanaan teks. Anda juga boleh menjana imej daripada penerangan teks. Imej sebagai modality berguna dalam MedTech, seni bina, pelancongan, pembangunan permainan, pemasaran, dan lain-lain. Dalam pelajaran ini, kami bekerja dengan model **GPT Image** hari ini di platform OpenAI.

## Matlamat pembelajaran

Menjelang akhir pelajaran ini anda akan dapat:

- Terangkan apa itu penjanaan imej dan di mana ia berguna.
- Fahami keluarga model `gpt-image` dan bagaimana ia berbeza daripada model DALL·E lama.
- Membangunkan aplikasi penjanaan imej dan mengedit imej.

## Apakah penjanaan imej?

Model penjanaan imej mencipta imej daripada arahan teks. Model moden seperti `gpt-image` mempelajari hubungan antara teks dan imej semasa latihan, kemudian secara berperingkat mengubah bunyi rawak menjadi imej yang sepadan dengan arahan anda.

Dua keluarga model imej yang dikenali adalah:

- **`gpt-image` (OpenAI)** — generasi terkini yang digunakan dalam pelajaran ini. Ia menyokong penjanaan teks-ke-imej dan penyuntingan imej (inpainting dengan topeng).
- **Midjourney** — model pihak ketiga yang popular dengan perkhidmatan dan aliran kerja berasaskan Discord sendiri.

> Model imej OpenAI yang lama — **DALL·E 2** dan **DALL·E 3** — adalah lama. DALL·E 3 tidak lagi tersedia untuk penempatan baharu, dan ciri seperti `create_variation` hanya wujud dalam DALL·E 2. Gunakan model `gpt-image` untuk aplikasi baharu.

> **Penting:** model `gpt-image` mengembalikan imej yang dijana sebagai **base64** (`b64_json`), bukan sebagai URL. Kod anda menukar rentetan base64 kepada bait dan menyimpannya — tiada URL imej untuk dimuat turun.


## Membina aplikasi penjanaan imej pertama anda

Jadi apa yang diperlukan untuk membina aplikasi penjanaan imej? Anda memerlukan perpustakaan berikut:

- **python-dotenv**, anda sangat disarankan menggunakan perpustakaan ini untuk menyimpan rahsia anda dalam fail *.env* yang berasingan dari kod.
- **openai**, perpustakaan ini adalah apa yang anda akan gunakan untuk berinteraksi dengan API OpenAI.
- **pillow**, untuk bekerja dengan imej dalam Python.


1. Cipta fail *.env* dengan kandungan berikut:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Kumpulkan perpustakaan di atas dalam fail bernama *requirements.txt* seperti berikut:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Seterusnya, buat persekitaran maya dan pasang perpustakaan tersebut:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTA]
> Untuk Windows, gunakan arahan berikut untuk membuat dan mengaktifkan persekitaran maya anda:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Tambahkan kod berikut dalam fail bernama *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Cipta objek OpenAI (membaca OPENAI_API_KEY dari .env anda)
    client = openai.OpenAI()


    try:
        # Cipta imej menggunakan API penjanaan imej
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Masukkan teks arahan anda di sini
            size='1024x1024',
            n=1
        )
        # Tetapkan direktori untuk imej yang disimpan
        image_dir = os.path.join(os.curdir, 'images')

        # Jika direktori tidak wujud, cipta ia
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Mula laluan imej (nota: jenis fail harus png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # model gpt-image mengembalikan imej sebagai base64 (b64_json), bukan URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Paparkan imej dalam penampil imej lalai
        image = Image.open(image_path)
        image.show()

    # tangkap pengecualian
    except openai.BadRequestError as err:
        print(err)

    ```

Mari kita terangkan kod ini:

- Pertama, kita mengimport perpustakaan yang kita perlukan, termasuk perpustakaan OpenAI, perpustakaan dotenv, modul base64, dan perpustakaan Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Selepas itu, kita mencipta klien, yang membaca kunci API dari ``.env`` anda.

    ```python
    # Cipta objek OpenAI
    client = openai.OpenAI()
    ```

- Seterusnya, kita menjana imej:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Masukkan teks arahan anda di sini
        size='1024x1024',
        n=1
    )
    ```

    Model `gpt-image` mengembalikan imej sebagai rentetan **base64** dalam `data[0].b64_json`. Kita mengekodnya ke bait dan menulisnya ke fail — tiada URL untuk dimuat turun.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Akhir sekali, kita membuka imej dan menggunakan penampil imej standard untuk memaparkannya:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Maklumat lanjut mengenai penjanaan imej

Mari lihat parameter `images.generate`:

- **model**, adalah model imej yang digunakan (contohnya `gpt-image-1`).
- **prompt**, adalah arahan teks yang digunakan untuk menjana imej. Di sini ia adalah "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, adalah saiz imej yang dijana (contohnya `1024x1024`, `1536x1024`, `1024x1536`, atau `"auto"`).
- **n**, adalah bilangan imej yang dijana. Di sini kita menjana satu.

> Model imej tidak mengambil parameter `temperature` — itu adalah kawalan untuk penjanaan teks. Untuk mendapatkan variasi, panggil API sekali lagi; untuk mengurangkan variasi, buat arahan anda lebih spesifik.

## Kebolehan tambahan penjanaan imej

Anda telah melihat bagaimana untuk menjana imej dengan beberapa baris Python. Model `gpt-image` juga boleh **mengedit** imej sedia ada. Dengan menyediakan imej, **masker** pilihan (yang menandakan kawasan untuk diubah), dan arahan, anda boleh mengubah sebahagian imej — contohnya, menambah topi pada arnab kami.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# suntingan juga dikembalikan sebagai base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Imej asas mengandungi hanya arnab; imej akhir mempunyai topi pada arnab itu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
